1. 라이브러리 설치

In [1]:
!pip install transformers pandas torch tqdm

2. Drive 모델 임베딩 추출 코드 실행

In [3]:
# 1. 필요한 라이브러리 임포트
import pandas as pd
import torch
from transformers import AutoTokenizer, EsmConfig, EsmForSequenceClassification
from tqdm.auto import tqdm
import numpy as np
from google.colab import drive
import os
import zipfile
import json
import safetensors.torch

# --- 0. Drive 연결 ---
drive.mount('/content/gdrive', force_remount=True)

# --- 1. 경로 설정 및 압축 파일 처리 (기존과 동일) ---
DACON_ROOT = '/content/gdrive/My Drive/DACON/MAI'
CHECKPOINT_PATH = os.path.join(DACON_ROOT, 'checkpoints', 'checkpoint-4000')
CONFIG_FILE_PATH = os.path.join(CHECKPOINT_PATH, 'config.json')
DATA_DIR = os.path.join(DACON_ROOT, 'data')

# (경로 출력 및 STEP 1 생략)


# --- 2. 모델 로드 (★최종 수정: Config 8192 재설정 및 가중치 수동 필터링 적용★) ---
print("\n[STEP 2/6] 최종 해결: Config 8192 설정 후, 크기 불일치 가중치 수동 필터링 시도...")
try:
    # 🌟🌟🌟 1. Config 파일 로드 및 EsmConfig 객체 생성 🌟🌟🌟
    if not os.path.exists(CONFIG_FILE_PATH):
        raise FileNotFoundError(f"🚨 config.json이 체크포인트 폴더에 없습니다. 파일을 확인해주세요.")

    with open(CONFIG_FILE_PATH, 'r', encoding='utf-8') as f:
        config_dict = json.load(f)

    config_dict['model_type'] = 'esm'
    config_dict['torch_dtype'] = 'float16'

    # 🚨🚨🚨 Config의 intermediate_size를 원본 모델 값인 8192로 재설정 🚨🚨🚨
    if 'intermediate_size' in config_dict and config_dict['intermediate_size'] != 8192:
         print(f"   -> [재설정] Config의 'intermediate_size'를 원본 값인 8192로 설정합니다.")
    config_dict['intermediate_size'] = 8192

    # Config 파일에 덮어쓰기
    with open(CONFIG_FILE_PATH, 'w', encoding='utf-8') as f:
        json.dump(config_dict, f, indent=4)

    config = EsmConfig.from_dict(config_dict)
    print("   -> EsmConfig 객체 생성 완료 (intermediate_size=8192).")

    # 🌟🌟🌟 2. 모델 인스턴스 수동 생성 및 토크나이저 로드 🌟🌟🌟
    model = EsmForSequenceClassification(config)
    tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_PATH, local_files_only=True)
    model.resize_token_embeddings(len(tokenizer))
    print(f"   -> 모델 임베딩 크기 조정 완료: {model.get_input_embeddings().weight.size(0)}")


    # 🌟🌟🌟 3. 가중치 로드 및 수동 필터링 (최종 해결 로직) 🌟🌟🌟
    model_file = os.path.join(CHECKPOINT_PATH, "model.safetensors")

    if os.path.exists(model_file):
        state_dict = safetensors.torch.load_file(model_file, device='cpu')
        print("   -> model.safetensors 파일 로드 성공.")
    else:
        model_file = os.path.join(CHECKPOINT_PATH, "pytorch_model.bin")
        if os.path.exists(model_file):
            state_dict = torch.load(model_file, map_location='cpu', weights_only=False)
            print("   -> pytorch_model.bin 파일 로드 성공.")
        else:
             raise FileNotFoundError(f"가중치 파일(model.safetensors 또는 pytorch_model.bin)을 {CHECKPOINT_PATH}에서 찾을 수 없습니다.")

    # 🚨🚨🚨 핵심 해결: 크기 불일치 가중치를 사전에 필터링 🚨🚨🚨
    model_state_dict = model.state_dict()
    filtered_state_dict = {}

    for name, param in state_dict.items():
        if name in model_state_dict and param.shape == model_state_dict[name].shape:
            filtered_state_dict[name] = param
        elif name in model_state_dict:
            # 크기 불일치 발생하는 가중치를 무시합니다. (예: 8192 vs 4096 모순)
            print(f"   -> [필터링] 크기 불일치 발생. 레이어 '{name}' 무시. 체크포인트: {param.shape}, 모델: {model_state_dict[name].shape}")
        else:
            # 모델에 없는 가중치(예: 이전의 잘못된 분류 헤드)를 무시합니다.
            print(f"   -> [필터링] 모델에 없는 레이어 '{name}' 무시.")

    # strict=False를 통해, 필터링 후 남아있는 Missing Keys(무시된 레이어)를 허용합니다.
    model.load_state_dict(filtered_state_dict, strict=False)

    print("✅ 모델 및 토크나이저 로드 성공 (모든 오류 해결 완료)!")

except Exception as e:
    print(f"🚨 [치명적 에러] 모델 로드에 최종적으로 실패했습니다. 오류: {e}")
    raise e


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- 3. 데이터 로드 (Drive에서 직접 로드) ---
print("\n[STEP 3/6] 데이터 로드 시작...")
try:
    test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
    sample_submission = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
    print("✅ 데이터 로드 성공: test.csv, sample_submission.csv")
except FileNotFoundError:
    print(f"🚨 [에러] Drive 경로 {DATA_DIR}에서 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    exit()

# --- 4. 임베딩 추출 함수 (기존과 동일) ---
def get_sequence_embedding(model, tokenizer, sequence):
    tokenizer.sep_token = tokenizer.sep_token if tokenizer.sep_token else '[SEP]'
    tokenizer.eos_token = tokenizer.eos_token if tokenizer.eos_token else '[EOS]'

    encoded_input = tokenizer(
        sequence,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512
    )

    with torch.no_grad():
        output = model(
            **encoded_input.to(model.device),
            output_hidden_states=True
        )

    last_hidden_state = output.hidden_states[-1]

    attn_mask = encoded_input['attention_mask'].unsqueeze(-1).to(model.device)
    summed = (last_hidden_state * attn_mask).sum(dim=1)
    counts = attn_mask.sum(dim=1).clamp(min=1)
    seq_emb = summed / counts

    return seq_emb.cpu()

# --- 5. 임베딩 추출 실행 ---
test_sequences = test_df['seq'].tolist()
all_embs = []
BATCH_SIZE = 32

print(f"\n[STEP 4/6] 임베딩 추출 시작. 총 {len(test_sequences)}개 서열 처리.")
for i in tqdm(range(0, len(test_sequences), BATCH_SIZE)):
    batch_sequences = test_sequences[i:i+BATCH_SIZE]
    emb = get_sequence_embedding(model, tokenizer, batch_sequences)
    all_embs.append(emb)

emb = torch.vstack(all_embs).float()
N, H = emb.shape
print(f"\n🎉 [STEP 5/6] 임베딩 추출 완료. 최종 크기 = {N} x {H}")

# --- 6. 제출 파일 생성 ---
print("\n[STEP 6/6] 제출 파일 생성 시작...")
emb_np = emb.numpy()
emb_cols = [f"emb_{i:04d}" for i in range(emb_np.shape[1])]

submission_df = sample_submission[['ID']].copy()
emb_df = pd.DataFrame(emb_np, columns=emb_cols)

submission = pd.concat([submission_df, emb_df], axis=1)
SUBMISSION_FILENAME = 'fine_tuned_4000_submission.csv'
submission.to_csv(SUBMISSION_FILENAME, index=False)
print(f"🔥 최종 제출 파일 생성 완료: {SUBMISSION_FILENAME}")

Mounted at /content/gdrive

[STEP 2/6] 최종 해결: Config 8192 설정 후, 크기 불일치 가중치 수동 필터링 시도...
   -> [재설정] Config의 'intermediate_size'를 원본 값인 8192로 설정합니다.
   -> EsmConfig 객체 생성 완료 (intermediate_size=8192).
   -> 모델 임베딩 크기 조정 완료: 4109
   -> model.safetensors 파일 로드 성공.
   -> [필터링] 모델에 없는 레이어 'esm.embeddings.position_embeddings.weight' 무시.
   -> [필터링] 크기 불일치 발생. 레이어 'esm.encoder.layer.0.output.dense.weight' 무시. 체크포인트: torch.Size([1024, 4096]), 모델: torch.Size([1024, 8192])
   -> [필터링] 크기 불일치 발생. 레이어 'esm.encoder.layer.1.output.dense.weight' 무시. 체크포인트: torch.Size([1024, 4096]), 모델: torch.Size([1024, 8192])
   -> [필터링] 크기 불일치 발생. 레이어 'esm.encoder.layer.10.output.dense.weight' 무시. 체크포인트: torch.Size([1024, 4096]), 모델: torch.Size([1024, 8192])
   -> [필터링] 크기 불일치 발생. 레이어 'esm.encoder.layer.11.output.dense.weight' 무시. 체크포인트: torch.Size([1024, 4096]), 모델: torch.Size([1024, 8192])
   -> [필터링] 크기 불일치 발생. 레이어 'esm.encoder.layer.12.output.dense.weight' 무시. 체크포인트: torch.Size([1024, 4096]), 모델: torch.Size([102

  0%|          | 0/429 [00:00<?, ?it/s]


🎉 [STEP 5/6] 임베딩 추출 완료. 최종 크기 = 13711 x 1024

[STEP 6/6] 제출 파일 생성 시작...
🔥 최종 제출 파일 생성 완료: fine_tuned_4000_submission.csv


In [4]:
import os

# --- 경로 설정 (이전 코드에서 사용된 경로 재사용) ---
DACON_ROOT = '/content/gdrive/My Drive/DACON/MAI'
SUBMISSION_FILENAME = 'fine_tuned_4000_submission.csv'

# Colab 환경의 현재 파일 경로
LOCAL_PATH = SUBMISSION_FILENAME
# Drive에 저장할 최종 경로
DRIVE_PATH = os.path.join(DACON_ROOT, SUBMISSION_FILENAME)

print(f"✅ 제출 파일 복사 시작: {LOCAL_PATH} -> Drive ({DRIVE_PATH})...")

# 쉘 명령어 'cp'를 사용하여 파일 복사 실행
# (Colab 환경에서 가장 쉽고 빠르게 파일을 복사하는 방법입니다.)
!cp {LOCAL_PATH} "{DRIVE_PATH}"

print("🎉 Drive 복사 완료! Google Drive의 'DACON/MAI' 폴더에서 파일을 확인해주세요.")

✅ 제출 파일 복사 시작: fine_tuned_4000_submission.csv -> Drive (/content/gdrive/My Drive/DACON/MAI/fine_tuned_4000_submission.csv)...
🎉 Drive 복사 완료! Google Drive의 'DACON/MAI' 폴더에서 파일을 확인해주세요.
